In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import torch

from src.config import SEED
from src.transfer import build_feature_extractor, build_fine_tuner, train_transfer

torch.manual_seed(SEED)

# Transfer Learning con ResNet50
## Fase 4 - Feature Extraction y Fine-tuning

En lugar de entrenar una red desde cero como la CNN, aprovechamos **ResNet50 preentrenada en ImageNet**: una red que ya aprendió a reconocer bordes, texturas y estructuras visuales complejas en 1.2 millones de imágenes.

Usamos ImageNet específicamente porque normalizamos nuestras imágenes con sus stats (mean y std), lo que garantiza que la red recibe los datos en el rango que espera.

Comparamos dos estrategias:

| Estrategia | Qué se entrena | LR | Parámetros entrenables |
|---|---|---|---|
| **Feature extraction** | Solo clasificador (fc) | 1e-3 | aprox 2049 |
| **Fine-tuning** | layer4 + clasificador | 1e-4 | aprox 15M |

**¿Por qué fine-tuning debería ganar?**  
Detectar imágenes IA requiere reconocer artefactos de difusión y texturas sintéticas que no existen en ImageNet. El feature extraction usa los features genéricos tal como vienen; el fine-tuning adapta las capas más profundas a nuestra tarea específica.

In [ ]:
model_fe = build_feature_extractor()

total = sum(p.numel() for p in model_fe.parameters())
trainable = sum(p.numel() for p in model_fe.parameters() if p.requires_grad)
print(f"Parámetros totales:      {total:,}")
print(f"Parámetros entrenables:  {trainable:,}  ({100*trainable/total:.1f}%)")

## 1. Feature Extraction

In [ ]:
historial_fe = train_transfer(model_fe, save_name="fe")

## 2. Fine-tuning

Partimos del mismo ResNet50 preentrenado pero descongelamos **layer4** (el último bloque residual) además del clasificador. El resto del backbone sigue congelado.

Usamos **LR = 1e-4** (10x más chico que en feature extraction) para actualizar los pesos preentrenados con pasos pequeños y no destruir los features que ya aprendió ResNet50.

**¿Por qué solo layer4?**
- layer1–3 detectan features genéricos (bordes, gradientes, texturas simples) que transfieren bien sin modificar
- layer4 detecta estructuras de alto nivel que necesitan adaptarse a los artefactos específicos de imágenes IA

In [ ]:
model_ft = build_fine_tuner()

total = sum(p.numel() for p in model_ft.parameters())
trainable = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
print(f"Parámetros totales:      {total:,}")
print(f"Parámetros entrenables:  {trainable:,}  ({100*trainable/total:.1f}%)")

print("\nCapas descongeladas:")
for name, param in model_ft.named_parameters():
    if param.requires_grad:
        print(f"  {name}")

In [ ]:
historial_ft = train_transfer(model_ft, save_name="ft", lr=1e-4)